In [ ]:
from pathlib import Path
import os
import sys
import warnings
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
from arch import arch_model
from arch.univariate import SkewStudent
from IPython.display import display

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config/paper.json').is_file())
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from finder_core.public_outputs import sequential_thresholds


In [ ]:
fit_gjr = False
markets = {'^GSPC': 'SP500', 'FTSE100': 'FTSE', 'GDAXI': 'DAX', 'CACT': 'CAC', 'STOXX': 'STOXX', 'CBU0.L': 'CBU0'}
TEST_CUTOFF = pd.Timestamp('2025-03-01')
MIN_TRAIN_OBSERVATIONS = 252
REFIT_EVERY = 21
MAX_OPTIMIZER_ITERATIONS = 3000
output = Path('Benchmark/Official/gjr_predictions.csv')


In [ ]:
def fit_gjr_skewt(returns_pct: pd.Series) -> Dict[str, float]:
    """Fit GJR-GARCH(1,1)-skewt to returns known at the forecast origin.

    Specification:
        r_t = mu + epsilon_t, epsilon_t = sigma_t * z_t
        sigma_t^2 = omega + alpha*epsilon_(t-1)^2
                   + gamma*I(epsilon_(t-1)<0)*epsilon_(t-1)^2
                   + beta*sigma_(t-1)^2
        z_t follows Hansen's standardized skewed Student-t distribution.

    Returns are in percentage points (e.g. 1% -> 1.0), not decimal units.
    """
    y = pd.Series(returns_pct, dtype=float).replace([np.inf, -np.inf], np.nan).dropna()
    if len(y) < MIN_TRAIN_OBSERVATIONS:
        raise ValueError(f'Only {len(y)} usable return observations; need at least {MIN_TRAIN_OBSERVATIONS}. Reduce MIN_TRAIN_OBSERVATIONS only if justified.')
    model = arch_model(y, mean='Constant', vol='GARCH', p=1, o=1, q=1, power=2.0, dist='skewt', rescale=False)
    result = model.fit(disp='off', show_warning=False, options={'maxiter': MAX_OPTIMIZER_ITERATIONS})
    convergence_flag = int(getattr(result, 'convergence_flag', 0))
    if convergence_flag != 0:
        raise RuntimeError(f'GJR-GARCH optimizer did not converge (convergence_flag={convergence_flag}). Do not silently report this fit; inspect the sample and optimizer settings.')
    params = result.params
    required = ['mu', 'omega', 'alpha[1]', 'gamma[1]', 'beta[1]', 'eta', 'lambda']
    missing = [name for name in required if name not in params.index]
    if missing:
        raise KeyError(f'Unexpected arch parameter names; missing {missing}; found {list(params.index)}')
    residuals = pd.Series(result.resid).replace([np.inf, -np.inf], np.nan).dropna()
    conditional_vol = pd.Series(result.conditional_volatility).replace([np.inf, -np.inf], np.nan).dropna()
    if residuals.empty or conditional_vol.empty:
        raise RuntimeError('The fitted model did not return usable residuals/conditional volatility.')
    fitted = {name: float(params[name]) for name in required}
    fitted['last_residual'] = float(residuals.iloc[-1])
    fitted['last_sigma2'] = float(conditional_vol.iloc[-1] ** 2)
    fitted['convergence_flag'] = convergence_flag
    skewt = SkewStudent()
    fitted['negative_partial_second_moment'] = float(skewt.partial_moment(2, z=0.0, parameters=[fitted['eta'], fitted['lambda']]))
    fitted['persistence'] = fitted['alpha[1]'] + fitted['gamma[1]'] * fitted['negative_partial_second_moment'] + fitted['beta[1]']
    fitted['n_fit_obs'] = int(len(y))
    return fitted

def one_step_variance(fit: Dict[str, float], previous_residual: float, previous_sigma2: float) -> float:
    """GJR one-step conditional variance, in squared percentage points."""
    eps2 = float(previous_residual) ** 2
    value = fit['omega'] + fit['alpha[1]'] * eps2 + fit['gamma[1]'] * eps2 * float(previous_residual < 0.0) + fit['beta[1]'] * float(previous_sigma2)
    if not np.isfinite(value) or value <= 0.0:
        raise FloatingPointError(f'Invalid one-step conditional variance: {value}')
    return float(value)

def forecast_test_period(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Generate strictly one-step-ahead conditional density forecasts in test period."""
    data = df.copy().reset_index(drop=True)
    data['_return_pct'] = data['_log_return_decimal'] * 100.0
    data['_threshold_pct'] = data['event_threshold_decimal'] * 100.0
    test_positions = np.flatnonzero(data['date'].to_numpy() >= evaluation_cutoff.to_datetime64())
    original_positions = np.flatnonzero(data['date'].to_numpy() >= TEST_CUTOFF.to_datetime64())
    refit_origin = int(original_positions[0])
    if len(test_positions) == 0:
        raise ValueError(f'No target observations on/after TEST_CUTOFF={TEST_CUTOFF.date()}.')
    forecasts: List[dict] = []
    fit_history: List[dict] = []
    fit_state: Optional[Dict[str, float]] = None
    last_successful_fit_end: Optional[pd.Timestamp] = None
    for test_count, target_pos in enumerate(test_positions):
        target_date = pd.Timestamp(data.loc[target_pos, 'date'])
        estimation_returns = data.loc[:target_pos - 1, '_return_pct'].dropna()
        if len(estimation_returns) < MIN_TRAIN_OBSERVATIONS:
            forecasts.append({'date': target_date, 'actual': int(data.loc[target_pos, 'crash_label']), 'actual_logreturn': data.loc[target_pos, '_log_return_decimal'], 'event_threshold': data.loc[target_pos, 'event_threshold_decimal'], 'garch_probability': np.nan, 'conditional_mean_pct': np.nan, 'conditional_variance_pct2': np.nan, 'conditional_volatility_pct': np.nan, 'conditional_volatility_ann': np.nan, 'realized_innovation_pct': np.nan, 'fit_end_date': pd.NaT, 'refit_performed': False, 'fit_observations': len(estimation_returns)})
            continue
        do_refit = fit_state is None or (target_pos >= refit_origin and (target_pos - refit_origin) % REFIT_EVERY == 0)
        if do_refit:
            try:
                fit_state = fit_gjr_skewt(estimation_returns)
                last_successful_fit_end = pd.Timestamp(data.loc[:target_pos - 1, 'date'].iloc[-1])
                fit_history.append({'forecast_date': target_date, 'fit_end_date': last_successful_fit_end, 'fit_observations': fit_state['n_fit_obs'], 'mu': fit_state['mu'], 'omega': fit_state['omega'], 'alpha': fit_state['alpha[1]'], 'gamma': fit_state['gamma[1]'], 'beta': fit_state['beta[1]'], 'eta': fit_state['eta'], 'lambda': fit_state['lambda'], 'persistence': fit_state['persistence'], 'convergence_flag': fit_state['convergence_flag']})
            except Exception as exc:
                if fit_state is None:
                    raise RuntimeError(f'Initial GJR-GARCH fit failed at {target_date.date()}: {exc}') from exc
                warnings.warn(f'GJR-GARCH refit failed at {target_date.date()}; retaining the last successful parameter/state estimate. Details: {exc}')
                do_refit = False
        sigma2_t = one_step_variance(fit_state, fit_state['last_residual'], fit_state['last_sigma2'])
        sigma_t = float(np.sqrt(sigma2_t))
        mu_t = float(fit_state['mu'])
        threshold_pct = data.loc[target_pos, '_threshold_pct']
        threshold_pct = float(threshold_pct) if pd.notna(threshold_pct) else np.nan
        if np.isfinite(threshold_pct):
            z_threshold = (threshold_pct - mu_t) / sigma_t
            skewt = SkewStudent()
            p_event = float(skewt.cdf(np.asarray([z_threshold], dtype=float), parameters=[fit_state['eta'], fit_state['lambda']])[0])
            p_event = float(np.clip(p_event, 0.0, 1.0))
        else:
            p_event = np.nan
        actual_return_pct = data.loc[target_pos, '_return_pct']
        actual_return_pct = float(actual_return_pct) if pd.notna(actual_return_pct) else np.nan
        actual_innovation_pct = actual_return_pct - mu_t if np.isfinite(actual_return_pct) else np.nan
        forecasts.append({'date': target_date, 'actual': int(data.loc[target_pos, 'crash_label']), 'actual_logreturn': data.loc[target_pos, '_log_return_decimal'], 'event_threshold': data.loc[target_pos, 'event_threshold_decimal'], 'garch_probability': p_event, 'conditional_mean_pct': mu_t, 'conditional_variance_pct2': sigma2_t, 'conditional_volatility_pct': sigma_t, 'conditional_volatility_ann': sigma_t * np.sqrt(252.0) / 100.0, 'realized_innovation_pct': actual_innovation_pct, 'fit_end_date': last_successful_fit_end, 'refit_performed': bool(do_refit), 'fit_observations': fit_state['n_fit_obs'], 'gjr_omega': fit_state['omega'], 'gjr_alpha': fit_state['alpha[1]'], 'gjr_gamma': fit_state['gamma[1]'], 'gjr_beta': fit_state['beta[1]'], 'skewt_eta': fit_state['eta'], 'skewt_lambda': fit_state['lambda'], 'gjr_persistence': fit_state['persistence']})
        if np.isfinite(actual_return_pct):
            fit_state['last_residual'] = actual_return_pct - mu_t
            fit_state['last_sigma2'] = sigma2_t
    forecast_df = pd.DataFrame(forecasts).sort_values('date').reset_index(drop=True)
    fit_history_df = pd.DataFrame(fit_history)
    return (forecast_df, fit_history_df)

In [ ]:
if fit_gjr:
    fitted_frames = []
    for market, code in markets.items():
        indicators = pd.read_csv(f'Benchmark/Data/Indicators_crash/{code}_df_Indicators.csv', parse_dates=['Date'])
        labels = pd.read_csv(f'Benchmark/Data/Indicators_crash/{code}_df_Crash_Label.csv', parse_dates=['date']).dropna(subset=['crash_label'])
        data = indicators.rename(columns={'Date': 'date'}).merge(labels, on='date', validate='one_to_one').sort_values('date')
        data['_log_return_decimal'] = data.logreturn if 'logreturn' in data else np.log(data.close).diff()
        data['event_threshold_decimal'] = data.crash_threshold
        finder_dates = pd.read_csv('FINDER/Official/daily_scores_all_markets.csv')
        evaluation_cutoff = pd.to_datetime(finder_dates.loc[finder_dates.market.eq(market), 'information_cutoff']).min()
        forecasts, fit_history = forecast_test_period(data)
        forecasts['market'] = market
        forecasts = forecasts.rename(columns={'date': 'information_cutoff', 'conditional_variance_pct2': 'cutoff_variance_pct2', 'realized_innovation_pct': 'cutoff_innovation_pct'})
        fitted_frames.append(forecasts)
    saved = pd.concat(fitted_frames, ignore_index=True)
else:
    saved = pd.read_csv(output, parse_dates=['information_cutoff'])


In [ ]:
frames = []
finder = pd.read_csv('FINDER/Official/daily_scores_all_markets.csv')
for market, code in markets.items():
    frame = saved.loc[saved.market.eq(market)].copy()
    labels = pd.read_csv(f'Benchmark/Data/Indicators_crash/{code}_df_Crash_Label.csv', parse_dates=['date']).set_index('date')
    prices = pd.read_csv(f'Benchmark/Data/OHLCV/{code}_df_OHLCV.csv', parse_dates=['Date']).sort_values('Date')
    next_dates = pd.Series(prices.Date.shift(-1).to_numpy(), index=prices.Date)
    frame['target_date'] = next_dates.reindex(frame.information_cutoff).to_numpy()
    frame['actual'] = labels.crash_label.reindex(frame.information_cutoff).to_numpy()
    frame['event_threshold'] = labels.crash_threshold.reindex(frame.information_cutoff).to_numpy()
    frame['actual_logreturn'] = labels.min_future_logret.reindex(frame.information_cutoff).to_numpy()
    eps2 = frame.cutoff_innovation_pct ** 2
    frame['conditional_variance_pct2'] = frame.gjr_omega + frame.gjr_alpha * eps2 + frame.gjr_gamma * eps2 * (frame.cutoff_innovation_pct < 0) + frame.gjr_beta * frame.cutoff_variance_pct2
    assert frame.conditional_variance_pct2.gt(0).all()
    frame['conditional_volatility_pct'] = np.sqrt(frame.conditional_variance_pct2)
    frame['conditional_volatility_ann'] = frame.conditional_volatility_pct * np.sqrt(252) / 100
    z = (frame.event_threshold * 100 - frame.conditional_mean_pct) / frame.conditional_volatility_pct
    frame['garch_probability'] = [float(SkewStudent().cdf(value, parameters=[eta, skew])) for value, eta, skew in zip(z, frame.skewt_eta, frame.skewt_lambda)]
    frame['realized_innovation_pct'] = frame.actual_logreturn * 100 - frame.conditional_mean_pct
    frame = frame.dropna(subset=['target_date', 'actual', 'garch_probability']).sort_values('target_date')
    assert (pd.to_datetime(frame.fit_end_date) < frame.information_cutoff).all()
    frame = frame[frame.target_date.dt.strftime('%Y-%m-%d').isin(finder.loc[finder.market.eq(market), 'target_date'])].copy()
    frame['threshold'] = sequential_thresholds(frame.garch_probability, frame.actual, frame.target_date, frame.information_cutoff)
    frame['prediction'] = (frame.garch_probability >= frame.threshold).astype(int)
    frame = frame.drop(columns=['fixed_threshold', 'fixed_prediction', 'sequential_threshold', 'sequential_prediction'], errors='ignore')
    frames.append(frame)
predictions = pd.concat(frames, ignore_index=True)
predictions.to_csv(output, index=False)
display(predictions.groupby('market').agg(n=('actual', 'size'), events=('actual', 'sum')))
